# ai05 Walkthrough — APIs & Random Forest

## Lesson ai05: APIs & Random Forest
### Sub-Lesson 05a: Fetching Live Data with APIs
### Sub-Lesson 05b: Building Better Models with Random Forest

**Purpose:** Learn to fetch live data from APIs AND build ensemble models that outperform single trees.

**ODE Competencies:** 2.14.1, 2.14.5, 5.1.2

In [ ]:
import requests                              # Fetch data from the internet
import pandas as pd                          # Work with DataFrames
import numpy as np                           # Numerical operations
import json                                  # Parse JSON
import warnings
warnings.filterwarnings("ignore")

print("✓ Libraries imported")

---

# PART 1: APIs — Getting Live Data

## What is an API?

**API = Application Programming Interface**

A contract between your code and a server. You ask for data in a specific format, server sends it back.

Think of it like a restaurant:
- **Menu** = API documentation (what is available?)
- **Your order** = HTTP request (what do you want?)
- **Waiter** = API endpoint (the middleman)
- **Kitchen** = Server/database (where the magic happens)
- **Your food** = JSON response (structured data)

**Key insight:** You do not go into the kitchen yourself. You use the waiter (API) as an intermediary.

## URL Structure

```
https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&minmagnitude=5.0&limit=10
Protocol         Host                    Path              Parameters
```

**Parameter rules:**
- `?` starts the parameters
- `key=value` pairs
- `&` separates multiple parameters

In [ ]:
# Step 1: Define the base URL for the weather API
baseUrl = "___"

# Step 2: Define parameters (latitude, longitude, daily metrics)
weatherParams = {
    "latitude": ___,                         # Medina, OH latitude
    "longitude": ___,                        # Medina, OH longitude
    # Your code here — add "daily" and "temperature_unit" keys
}

# Step 3: Send GET request using requests.get()
# Your code here
print(f"Status code: {___}")  # What status code did we get?

# Step 4: Convert response to Python dict
# Your code here
print("✓ Data fetched successfully")

### Try This:

What happens if you change the latitude to 99.99 (invalid)? What status code do you get?

In [ ]:
# Try it here:
# Your code here

In [ ]:
# Extract the daily forecast data (nested inside weatherData)
dailyData = weatherData["daily"]

# Create DataFrame from the nested dictionaries
weatherDf = pd.DataFrame({
    "date": dailyData["time"],                             # List of date strings
    "temp_max": dailyData["temperature_2m_max"],           # Max temperature each day
    "temp_min": dailyData["temperature_2m_min"],           # Min temperature each day
    "precipitation": dailyData["precipitation_sum"]         # Total precipitation each day
})

# Convert date column to datetime objects for proper sorting/filtering
weatherDf["date"] = pd.to_datetime(weatherDf["date"])

print("Weather Data for Medina, OH:")
print(weatherDf.head(10))

In [ ]:
# Build earthquake API query
eqBaseUrl = "https://earthquake.usgs.gov/fdsnws/event/1/query"

eqParams = {
    "format": "___",                        # Response format (try "geojson")
    "minmagnitude": ___,                    # Only earthquakes >= this magnitude
    "limit": ___                            # Return at most this many results
}

# Step 1: Fetch the data from USGS API
# Your code here
# eqResponse = ...
# eqData = ...

# Step 2: Parse GeoJSON response
earthquakes = []
for feature in eqData["features"]:
    props = feature["properties"]                # Metadata (magnitude, place, etc.)
    coords = feature["geometry"]["coordinates"]  # [longitude, latitude, depth]
    # Your code here — create a dict with magnitude, place, longitude, latitude, depth_km
    # earthquakes.append({...})

# Convert to DataFrame
eqDf = pd.DataFrame(earthquakes)
print("Recent Major Earthquakes:")
print(eqDf.to_string())

---

# PART 2: RANDOM FOREST — ENSEMBLE LEARNING

## Random Forest Overview

**Random Forest** = Collection of decision trees trained on random data + random features.

### Two Layers of Randomness:
1. **Random Samples:** Each tree on different bootstrap sample (~500 rows WITH replacement)
2. **Random Features:** At each split, only consider random subset of features

### Prediction:
- All 100 trees make a prediction
- **Majority vote wins**
- Final prediction = most common answer

In [ ]:
from sklearn.ensemble import RandomForestClassifier   # Ensemble model with multiple trees
from sklearn.tree import DecisionTreeClassifier       # Single tree for comparison
from sklearn.model_selection import train_test_split  # Split into train/test
from sklearn.metrics import accuracy_score, classification_report  # Evaluate predictions

# Load NBA game data
nbaData = pd.read_csv("nba_win_prediction.csv")

print(f"Shape: {nbaData.shape}")                                          # (games, features)
print(f"Columns: {list(nbaData.columns)}")                                # What features we have
print(f"Baseline accuracy: {(nbaData['home_win'] == 1).mean():.1%}")      # % games where home wins

In [ ]:
# Define which columns are features (X) vs target (y)
featureColumns = ["pts_diff", "reb_diff", "ast_diff", "fg_pct_diff", "tov_diff", "stl_diff"]
X = nbaData[___]                  # Features: differences between teams
y = nbaData[___]                  # Target: 1 if home team won, 0 if away won

# Split into training (80%) and test (20%) sets
# Your code here — use train_test_split()
# xTrain, xTest, yTrain, yTest = ...

print(f"Training set: {xTrain.shape[0]} games")
print(f"Test set: {xTest.shape[0]} games")

In [ ]:
# Create Random Forest with 100 decision trees
# Your code here — create RandomForestClassifier with n_estimators=100, random_state=42
# forestModel = ...
# forestModel.fit(xTrain, yTrain)

# Make predictions on test set
# Your code here
# yPred = ...

# Calculate accuracy (% of predictions that match actual outcome)
# Your code here
# forestAccuracy = ...

print(f"\nRandom Forest Accuracy: {forestAccuracy:.2%}")
print(f"Baseline (always home): 54%")
print(f"Improvement: +{(forestAccuracy - 0.54):.2%}")

In [ ]:
# Extract feature importance from the trained forest
# Each feature gets a score based on how much it helped trees make better splits
featureImportance = pd.DataFrame({
    "Feature": ___,                          # Your code here
    "Importance": ___                        # Your code here
}).sort_values("Importance", ascending=False)

print("\nFeature Importance Ranking:")
print(featureImportance.to_string(index=False))

In [ ]:
# Train a SINGLE decision tree for comparison
# Your code here — create DecisionTreeClassifier with random_state=42
# singleTree = ...
# singleTree.fit(xTrain, yTrain)

# Make predictions on same test set
# Your code here
# yPredSingle = ...
# singleAccuracy = ...

print(f"\nComparison:")
print(f"Single Tree: {singleAccuracy:.2%}")              # One expert
print(f"Random Forest (100 trees): {forestAccuracy:.2%}")  # 100 experts
print(f"Improvement: +{(forestAccuracy - singleAccuracy):.2%}")  # How much better the forest is

### Try This:

What happens if you change `n_estimators` to 10? What about 1000? How does accuracy change?

In [ ]:
# Experiment here:
# Your code here

## Summary

### Part 1: APIs
✓ APIs = structured data contracts
✓ URL structure: protocol + host + endpoint + parameters
✓ JSON is nested dicts/lists
✓ requests.get() + .json() = fetch + parse

### Part 2: Random Forest
✓ Ensemble = many experts > one expert
✓ Two randomness layers: data + features
✓ Majority voting combines predictions
✓ Feature importance shows what matters
✓ Better accuracy than single trees